# SnapUGC-LightKD Final 1000-Video Run

This notebook runs the clean v2 thesis pipeline:

- DOVER quality scoring
- CLIP/R(2+1)D/YAMNet/Sentence-T5/Qwen2.5-VL feature extraction
- Temporal Teacher, Student baseline, and Student+KD training

In [ ]:
# 1. Install dependencies
!pip install -q -U "transformers>=4.49.0" accelerate sentence-transformers qwen-vl-utils av
!pip install -q eva-decord tensorflow tensorflow_hub scipy pandas tqdm matplotlib

import os, sys, glob, json, torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
# 2. Paths
MAX_VIDEOS = 1000
RUN_DOVER = True
RUN_QWEN = True

INPUT_CANDIDATES = [
    '/kaggle/input/datasets/nguyntuncng/snapugc-dataset',
    '/kaggle/input/snapugc-dataset',
]
INPUT_DIR = next((p for p in INPUT_CANDIDATES if os.path.exists(p)), INPUT_CANDIDATES[-1])
TRAIN_CSV = os.path.join(INPUT_DIR, 'train_data.csv')
TRAIN_VIDEO_DIR = os.path.join(INPUT_DIR, 'train_videos')

REPO_CANDIDATES = [
    '/kaggle/working/SnapUGC-LightKD',
    '/kaggle/input/snapugc-lightkd/SnapUGC-LightKD',
    '/kaggle/input/snapugc-lightkd',
]
REPO_DIR = next((p for p in REPO_CANDIDATES if os.path.exists(os.path.join(p, 'scripts'))), None)
if REPO_DIR is None:
    raise FileNotFoundError('Cannot find SnapUGC-LightKD. Upload/copy the repo and update REPO_DIR.')

OUTPUT_DIR = '/kaggle/working/final_1000_videos'
FEATURES_PATH = os.path.join(OUTPUT_DIR, 'features_final_1000.json')
RESULTS_DIR = os.path.join(OUTPUT_DIR, 'results')
DOVER_OUT = os.path.join(OUTPUT_DIR, 'dover')
DOVER_CSV = ''
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print('INPUT_DIR:', INPUT_DIR)
print('REPO_DIR:', REPO_DIR)
print('FEATURES_PATH:', FEATURES_PATH)

In [ ]:
# 3. DOVER quality scores
if RUN_DOVER:
    !git clone -q https://github.com/VQAssessment/DOVER.git /kaggle/working/DOVER 2>/dev/null || true
    %cd /kaggle/working/DOVER
    !pip install -q -r requirements.txt || true
    !python evaluate_a_set_of_videos.py -in "$TRAIN_VIDEO_DIR" -out "$DOVER_OUT"
    csvs = glob.glob(os.path.join(DOVER_OUT, '**', '*.csv'), recursive=True)
    print('DOVER csv candidates:', csvs[:5])
    DOVER_CSV = csvs[0] if csvs else ''
else:
    print('Skipping DOVER; neutral quality scores will be used.')

%cd /kaggle/working
print('DOVER_CSV:', DOVER_CSV)

In [ ]:
# 4. Feature extraction
cmd = [
    sys.executable, os.path.join(REPO_DIR, 'scripts/extract_features.py'),
    '--csv', TRAIN_CSV,
    '--videos', TRAIN_VIDEO_DIR,
    '--out', FEATURES_PATH,
    '--max', str(MAX_VIDEOS),
    '--save-every', '25',
    '--num-frames', '16',
    '--motion-clips', '4',
    '--motion-frames', '16',
    '--qwen-frames', '8',
]
if DOVER_CSV:
    cmd += ['--dover-csv', DOVER_CSV]
if not RUN_QWEN:
    cmd += ['--skip-qwen']
print(' '.join(cmd))
!{" ".join(cmd)}

In [ ]:
# 5. Validate feature schema
with open(FEATURES_PATH, 'r', encoding='utf-8') as f:
    features = json.load(f)
print('samples:', len(features))
first = features[0]
for key in ['clip_frame_embeddings', 'motion_clip_embeddings', 'dover_scores', 'yamnet_embedding_mean', 'metadata_text_embedding', 'caption_embedding', 'rationale_embedding', 'qwen_caption']:
    value = first.get(key)
    if isinstance(value, list):
        print(key, 'len:', len(value), 'nested:', len(value[0]) if value and isinstance(value[0], list) else '')
    else:
        print(key, value if key == 'dover_scores' else type(value))

In [ ]:
# 6. Train Teacher, Student baseline, and Student+KD
cmd = [
    sys.executable, os.path.join(REPO_DIR, 'scripts/train.py'),
    '--data', FEATURES_PATH,
    '--save-dir', RESULTS_DIR,
    '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
    '--teacher-epochs', '40',
    '--student-epochs', '60',
    '--batch', '16',
    '--teacher-hidden', '512',
    '--student-hidden', '256',
    '--teacher-blocks', '2',
    '--alpha', '0.5',
    '--beta', '0.3',
    '--attn-kd', '0.1',
    '--gamma', '0.2',
    '--delta', '0.2',
]
print(' '.join(cmd))
!{" ".join(cmd)}

In [ ]:
# 7. Show report
report_path = os.path.join(RESULTS_DIR, 'final_experiment_report.json')
with open(report_path, 'r') as f:
    report = json.load(f)
print(json.dumps(report, indent=2)[:5000])

In [ ]:
# 8. Package outputs
!zip -qr /kaggle/working/snapugc_lightkd_final_1000_outputs.zip "$OUTPUT_DIR"
print('/kaggle/working/snapugc_lightkd_final_1000_outputs.zip')